In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob
import joblib
from sklearn.metrics import r2_score

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
cities = [
    ('LosAngeles', 'US', 'logincome'),
    ('NewYork', 'US', 'logincome'),
    ('Chicago', 'US', 'logincome'),
    ('Philadelphia', 'US', 'logincome'),
    ('Boston', 'US', 'logincome'),
    ('SanFrancisco', 'US', 'logincome'),
    ('Miami', 'US', 'logincome'),
    ('Sydney', 'Australia', 'med_hhinc'),
    ('Melbourne', 'Australia', 'med_hhinc'),
    ('Brisbane', 'Australia', 'med_hhinc'),
    ('Perth', 'Australia', 'med_hhinc'),
    ('Adelaide', 'Australia', 'med_hhinc'),
    # ('HongKong', 'China', 'HK15')
]

In [ ]:
def compare_r2(city, country, target, raito, sample_ratio):
    # strategy
    df = pd.read_csv(f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy/results.csv')
    label_sdg_file = f"../../data/processed/0labels/{country}.csv"
    labels_sdg = pd.read_csv(label_sdg_file)
    df = pd.merge(df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    df = df[df['target'] == target]
    
    strategy_all_r2 = df[df['ratio'] == ratio]['R2'].values[0]
    
    file = f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy/results.h5'
    detail_df = pd.read_hdf(
                file,
                key=f'R{str(ratio).replace(".", "")}'
            )
    detail_df = detail_df[[target, f'pred_{target}']]
    
    sample_size = int(len(detail_df) * sample_ratio)
    
    detail_df = detail_df.sort_values(by=target, ascending=True)
    detail_df = detail_df.iloc[:sample_size]
    detail_df = detail_df.reset_index(drop=True)
    
    strategy_r2 = r2_score(detail_df[target], detail_df[f'pred_{target}'])
    # print(f"strategy_r2: {strategy_r2}")
    # print(f"strategy_all_r2: {strategy_all_r2}")
    
    # random
    baseline_df = pd.read_csv(f'../../data/regression_outputs_new/Ratio/{country}/{city}/Fuse/Multi_Concat/results.csv')
    baseline_df = pd.merge(baseline_df, labels_sdg[['ID', 'SDG']], left_on='target', right_on='ID')
    baseline_df = baseline_df[baseline_df['target'] == target]
    
    random_all_r2 = np.mean(baseline_df[baseline_df['ratio'] == ratio]['R2'].values)
    file = f'../../data/regression_outputs_new/Ratio/{country}/{city}/Fuse/Multi_Concat/results.h5'
    r2s = []
    for fold in range(10):
        detail_df = pd.read_hdf(
                    file,
                    key=f'R{str(ratio).replace(".", "")}-F{fold}'
                )
        detail_df = detail_df[[target, f'pred_{target}']]
        detail_df = detail_df.sort_values(by=target, ascending=True)
        detail_df = detail_df.iloc[:sample_size]
        detail_df = detail_df.reset_index(drop=True)
        r2s.append(r2_score(detail_df[target], detail_df[f'pred_{target}']))
    random_r2 = np.mean(r2s)
    # print(f"random_r2: {random_r2}")
    # print(f"random_all_r2: {random_all_r2}")
    return strategy_all_r2, strategy_r2, random_all_r2, random_r2

In [ ]:
res = []
for city, country, target in cities:
    # print(f"city: {city}, country: {country}")
    for ratio in np.arange(0.1, 1.0, 0.1):
        ratio = round(ratio, 1)
        # print(f"ratio: {ratio}")
        strategy_all_r2, strategy_r2, random_all_r2, random_r2 = compare_r2(city, country, target, ratio, 0.3)
        res.append([ratio, strategy_all_r2, strategy_r2, random_all_r2, random_r2, city, country])
res_df = pd.DataFrame(res, columns=['ratio', 'strategy_all_r2', 'strategy_r2', 'random_all_r2', 'random_r2', 'city', 'country'])
res_df

In [ ]:
res_df['r2_diff'] = res_df['strategy_r2'] - res_df['random_r2']
res_df['r2_diff_all'] = res_df['strategy_all_r2'] - res_df['random_all_r2']
res_df

In [ ]:
res_df['ratio_percent'] = res_df['ratio'] * 100

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Polish lineplot labels to be terse and professional.
sns.lineplot(data=res_df, x='ratio_percent', y='r2_diff', ax=ax, marker='o', color='#3973b8', 
             label='Low-income areas', linewidth=2)
sns.lineplot(data=res_df, x='ratio_percent', y='r2_diff_all', ax=ax, marker='o', color='#4faf69', 
             label='All areas', linewidth=2)

# Add reference line and refine the text.
ax.axhline(0, color='red', linestyle='--', linewidth=2)
ax.text(70, -0.07, 'Zero difference line', color='black', fontsize=24, ha='center', va='bottom')

# Keep only the left and bottom spines (unchanged).
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

# Polish axis labels.
ax.set_xlabel("Sampling ratio (%)", fontsize=24)
ax.set_ylabel("$R^2$ difference by income", fontsize=24)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)

ax.set_xlim(10, 95)

# Polish legend.
plt.legend(fontsize=24, title=None, frameon=False)
plt.savefig('../../data/figure_assets/sampling_income.svg', bbox_inches='tight')
plt.show()

In [ ]:
tmp = res_df[res_df['ratio'] == 0.6]
tmp

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

sns.scatterplot(data=tmp, x='random_r2', y='strategy_r2', ax=ax, color='#3973b8', s=100, label='Low-income areas')
sns.scatterplot(data=tmp, x='random_all_r2', y='strategy_all_r2', ax=ax, color='#4faf69', s=100, label='All areas')

# 1:1 line
x = np.linspace(0, 1, 100)
y = x
ax.plot(x, y, color='red', linestyle='--', linewidth=2)

# Keep only the left and bottom spines (unchanged).
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

# Polish axis labels.
ax.set_xlabel("$R^2$  in random sampling", fontsize=24)
ax.set_ylabel("$R^2$  in strategic sampling", fontsize=24)
ax.tick_params(axis='x', labelsize=22)
ax.tick_params(axis='y', labelsize=22)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Polish legend.
plt.legend(fontsize=24, title=None, frameon=False)
plt.savefig('../../data/figure_assets/sampling_income_6.svg', bbox_inches='tight')
plt.show()